Vamos ahora a tratar las imputaciones registradas con el sistema MES

En las imputaciones es donde esta el detalle real de ejecución (qué máquina, qué tiempos reales, quién y cuándo).

In [70]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [71]:
ruta_imputations = os.path.join("res\\raw", "imputaciones.csv")
print(ruta_imputations)

res\raw\imputaciones.csv


In [72]:
columns_imputations = ["FechaCreacionDte",	"smNodoId",	"smNodoNme",	"smOrdenId",	"smMaterialId",	"smOrdenNme",	"smFaseId",	"smFaseNme",	"CantidadManualOKNbr",	"CantidadManualNOKNbr",	"MinutosRealNbr",	"MinutosPreparacionNbr",	"MinutosProduccionNbr",	"PorcentajeDedicacionNbr",	"ProductivoInd",	"smIncidenciaId",	"smIncidenciaNme",	"smFasePersonalizado1Txt",	"smNodoId2",	"smNodoPersonalizado1Nbr",	"smNodoPersonalizado2Nbr",	"ssEnumFaseEstadoId",	"SmFaseUid",	"TraspasadoInd",	"JsonBodyTxt",	"JsonResponseTxt"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)

C:\Users\jaume\AppData\Local\Temp\ipykernel_18100\1223014052.py:6: DtypeWarning: Columns (0: smFaseId, 1: smIncidenciaId, 2: smIncidenciaNme) have mixed types. Specify dtype option on import or set low_memory=False.
  df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)


In [73]:
display(df_imputations.sample(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,CantidadManualOKNbr,CantidadManualNOKNbr,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,smFasePersonalizado1Txt,smNodoId2,smNodoPersonalizado1Nbr,smNodoPersonalizado2Nbr,ssEnumFaseEstadoId,SmFaseUid,TraspasadoInd,JsonBodyTxt,JsonResponseTxt
148889,2025-05-14 13:17:59.303,CM06,CV KONDIA HM-2010 CU,OF2504443,PE2XA3X0064,ALLONGE COUDÉE 103076282,8,PE01,0.0,NaN,79.13333,79.13333,0.0,100.0,1,NaN,NaN,NaN,CM06,NaN,NaN,FINALIZADA_OLANET,1867F285-853F-490B-A44A-83CC0B9E9565,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[]}"


In [74]:
display(df_imputations.shape)

(196576, 26)

## Limpieza de atributos
Eliminar del dataset aquellos atributos prescindibles. 


In [75]:
def eliminar_columnas(df):
    """
    Elimina de df_imputations las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original).
    """

    cols_sobran = [
        "JsonResponseTxt", "JsonBodyTxt", "TraspasadoInd", "ssEnumFaseEstadoId", 
        "smNodoPersonalizado2Nbr", "smNodoPersonalizado1Nbr", "smNodoId2",
        "smFasePersonalizado1Txt", "CantidadManualOKNbr",
        "CantidadManualNOKNbr", "smFaseId",
        
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_imputations]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_imputations.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [76]:
df_imputations_limpieza = eliminar_columnas(df_imputations)

Eliminadas 11 columnas. Quedan 15 columnas, 196576 filas.


In [77]:
display(df_imputations_limpieza.sample(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid
195057,2025-01-23 21:11:06.250,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2500091,PE1XA2X0007,Allonge coudée 103105144,DES,33.81667,0.0,33.81667,100.0,1,NaN,NaN,C77277B2-5F8A-49CD-A452-2CD5A377EA7F


### Limpieza por intervalo de fechas
Trabajamos dentro del rango entre junio 2025 y antes de agosto 2026.

In [78]:
def filtrar_desde_fecha(df, fecha_desde, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro >= {fecha_desde}.")

    limite = pd.to_datetime(fecha_desde)
    df_out = df_out[df_out[columna] >= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [79]:
df_imputations_limpieza = filtrar_desde_fecha(df_imputations_limpieza, "2025-06-01")

Partida: 196576 fases
Conservadas (desde 2025-06-01): 141329 (descartadas 55247)


In [80]:
def filtrar_hasta_fecha(df, fecha_hasta, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea =< fecha_hasta.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro =< {fecha_hasta}.")

    limite = pd.to_datetime(fecha_hasta)
    df_out = df_out[df_out[columna] <= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (hasta {fecha_hasta}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [81]:
df_imputations_limpieza = filtrar_hasta_fecha(df_imputations_limpieza, "2026-08-01")

Partida: 141329 fases
Conservadas (hasta 2026-08-01): 140485 (descartadas 844)


Voy a guarda la tabla de las imputaciones en `res`

In [82]:
import pyarrow as pa

#df_imputations_limpieza["smFaseId"] = df_imputations_limpieza["smFaseId"].astype(str)

df_imputations_limpieza.to_parquet("res/procesed/df_imputations_limpieza.parquet")

In [83]:
#break

## Filtrar imputaciones por maquinas productivas
Todas las imputaciones que no pertenezcan a la selección de maquinas productivas (`grupo_maquinas`) debo hacerlas desaparecer ya que son imputaciones a no emplear para mi proyecto. 

In [84]:
# cargo el df de maquinas y grupos
maquina_grupo = pd.read_parquet("res//procesed/maquina_grupo.parquet")
display(maquina_grupo.sample(1))
print(f"Total de máquinas: {len(maquina_grupo)}")
print(f"Grupos de máquinas: {len(maquina_grupo["CodigoGrupo"].unique())}")

#print(len(maquina_grupo["CodigoMaquina"].unique()))
#print(len(maquina_grupo["CodigoMaquina"]))

,IDMaquina,CodigoMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
23,3045d81e-eb32-400c-b70f-652e75db78b2,CM17,MATSUURA MX-330,ccbacbbe-ba64-469a-8cf5-4f2b9f027bd5,C5,Centre de Mecanitzat 5 Eixos petites


Total de máquinas: 32
Grupos de máquinas: 9


> Para comprender mejor la siguiente celda del código, quiero remarcar que para `maquinas_validas[CodigoMaquina]` tiene su codigo homologo en `df_imputations_limpieza[smNodoId]`. Esto permite relacionar facilmente ambas tamblas mediante el código de la máquina. 

In [85]:
maquinas_validas = maquina_grupo["CodigoMaquina"].unique()
print(f"Imputaciones antes: {len(df_imputations_limpieza)}")
df_imput_productivas = df_imputations_limpieza[
    df_imputations_limpieza["smNodoId"].isin(maquinas_validas)
]
print(f"Imputaciones después: {len(df_imput_productivas)}")
print(f"Eliminadas (imputaciones en máquinas no productivas): {len(df_imputations_limpieza) - len(df_imput_productivas)}")

Imputaciones antes: 140485
Imputaciones después: 59574
Eliminadas (imputaciones en máquinas no productivas): 80911


In [86]:
fuera = round(((len(df_imputations_limpieza)-len(df_imput_productivas))/len(df_imputations_limpieza))*100, 2)
print(f"Desaparecen el {fuera}% de las imputaciones porque pertenecen a máquinas fuera de la població de estudio")

Desaparecen el 57.59% de las imputaciones porque pertenecen a máquinas fuera de la població de estudio


## Resolver fases imputadas en N nodos
Interesa averiguar si se ha dado una misma fase en distintas maquinas, por ello debo verificar si todas las imputaciones de una misma fase se han hecho en la misma máquina.   
Para cada imputación podemos saber en que nodo se ha realizado, un nodo puede ser una maquina o un centro de trabajo.

In [87]:
maquinas_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()
print(f"Total de fases productivas: {len(df_imput_productivas.groupby('SmFaseUid'))}")
print("")
print("Entre las fases productivas, el número de máquinas distintas por fase es:")
print(maquinas_por_fase.value_counts())

Total de fases productivas: 19149

Entre las fases productivas, el número de máquinas distintas por fase es:
smNodoId
1    18250
2      860
3       37
4        2
Name: count, dtype: int64


Averiguar que maquinas son las que presentan imputaciones de una misma fase en distintos nodos es innecesario, lo que realmente me interesa es ver si este cambio de nodo traspasa entre distintos grupos de máquinas:

In [88]:
# recuento de nodos por fase (sobre imputaciones productivas)
nodos_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()

fases_mono = nodos_por_fase[nodos_por_fase == 1].index
fases_multi = nodos_por_fase[nodos_por_fase > 1].index

# --- 1. Peso en número de fases ---
total_fases = len(nodos_por_fase)
n_multi = len(fases_multi)
print(f"Fases totales:            {total_fases}")
print(f"Fases mono-nodo:          {len(fases_mono)} ({len(fases_mono)/total_fases:.1%})")
print(f"Fases multi-nodo:         {n_multi} ({n_multi/total_fases:.1%})")

# --- 2. Peso en tiempo real (¿son marginales también en volumen de trabajo?) ---
tiempo_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]["MinutosRealNbr"].sum()
tiempo_total = df_imput_productivas["MinutosRealNbr"].sum()
print(f"\nTiempo real en fases multi-nodo: {tiempo_multi/tiempo_total:.1%} del total")

Fases totales:            19149
Fases mono-nodo:          18250 (95.3%)
Fases multi-nodo:         899 (4.7%)

Tiempo real en fases multi-nodo: 15.6% del total


Hay pocas fases en multi-nodo (4,6%), pero con peso desproporcionado en el tiempo total de todas las imputaciones (14,5%). No son ruido marginal, sino fases más "pesadas" que la media.      
Entonces el 4,6% de las fases acumula el 14,5% del tiempo real. Eso significa que una fase multi-nodo tienen más tiempo real que una fase normal. Hablando con el equipo de producción tiene sentido lógico, son fases largas o complejas, y precisamente por eso se repartieron entre varias máquinas o sesiones.

> Voy a inspeccionarlas.

¿En que máquinas aparecen las fases multi nodo?

In [89]:
# ¿en qué máquinas aparecen las fases multi-nodo?
imput_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]

t = imput_multi.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()
fraccion_dominante = t.groupby("SmFaseUid").max() / t.groupby("SmFaseUid").sum()
print(f"Total de fases donde la maquina dominante concentra >80% del tiempo: {(fraccion_dominante > 0.8).mean():.1%}")

Total de fases donde la maquina dominante concentra >80% del tiempo: 33.5%


En el 35,6% de las fases multi-nodo la máquina dominante concentra más del 80% del tiempo, es decir, en el 63,2% de las imputaciones en `df_imput_productivas`, el tiempo está genuinamente repartido, la "maquina dominante" no domina tanto, convive con otras máquinas con peso real.  
  
> Voy a trabajar con ese 36,8% de fases donde es facilmente identificable la maquina dominante, para las 63,2% restantes voy a prescindir, obviando las mismas para este proyecto. 

In [90]:
# tiempo por (fase, nodo)
tiempo_fase_nodo = df_imput_productivas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()

# fracción de la dominante en cada fase
dominante = tiempo_fase_nodo.groupby("SmFaseUid").max()
total = tiempo_fase_nodo.groupby("SmFaseUid").sum()
fraccion_dominante = dominante / total

# fases válidas: la dominante concentra >80% (esto incluye automáticamente las mono-nodo, que son 100%)
fases_validas = fraccion_dominante[fraccion_dominante > 0.8].index

print(f"Fases totales antes:     {fraccion_dominante.shape[0]}")
print(f"Fases válidas (dominante >80%): {len(fases_validas)}")
print(f"Fases descartadas (repartidas): {fraccion_dominante.shape[0] - len(fases_validas)}")

Fases totales antes:     19149
Fases válidas (dominante >80%): 18548
Fases descartadas (repartidas): 601


#### Revisar incidencias
Hay imputaciones en maquinas que corresponden a incidencias, inspecciono cuantas son y planteo descartarlas, ya que estas no tocan tiempo productivo.

In [91]:
# Revisar incidencias:
df_incidencias = df_imput_productivas[df_imput_productivas["smIncidenciaId"].notnull()]  
print(df_incidencias.shape)

(4, 15)


In [92]:
pre_eliminar_incidencias = df_imput_productivas.copy()
print(f"Fases contando con incidencias: {len(pre_eliminar_incidencias)}")
df_imput_productivas = df_imput_productivas[df_imput_productivas["smIncidenciaId"].isnull()]
print(f"Fases sin contar con incidencias: {len(df_imput_productivas)}")
print(f"Total de incidencias eliminadas: {len(pre_eliminar_incidencias) - len(df_imput_productivas)}")

Fases contando con incidencias: 59574
Fases sin contar con incidencias: 59570
Total de incidencias eliminadas: 4


Entonces partimos de 140.485 imputaciones, el recorte a operaciones productivas elimina el 52,54%, dejando 66.668 imputaciones repartidas en 21830 fases productivas, de las cuales nos quedamos con 21186 donde podemos identificar una maquina dominante como la responsable de la realización del trabajo en la fase. 

## Tabla final imputaciones: una instancia por cada fase
Finalmente voy a construir la tabla, con una fila por fase válida, con su grupo con la máquina dominante, y sus tiempos reales agregados.

Agregación de imputaciones a nivel de fase

Los datos de ejecución procedentes del MES se registran a nivel de imputación, de modo que una misma fase de fabricación puede acumular múltiples imputaciones correspondientes a las distintas sesiones de trabajo, operarios o intervalos temporales en que se ejecutó. Sin embargo, la unidad de análisis de este estudio es la fase, no la imputación individual. Por ello, resulta necesario agregar las imputaciones de cada fase en un único registro que la represente de forma completa.

Esta agregación responde a dos motivos fundamentales. En primer lugar, la variable objetivo del estudio —el ratio entre el tiempo real y el tiempo teórico— exige que ambos tiempos estén definidos a un mismo nivel. El tiempo teórico se establece en el ERP a nivel de fase, por lo que el tiempo real debe consolidarse igualmente a nivel de fase para que la comparación entre ambos sea coherente; de lo contrario, se dispondría de múltiples tiempos reales enfrentados a un único tiempo teórico, imposibilitando el cálculo del ratio.

En segundo lugar, el modelo que se pretende desarrollar opera sobre la unidad de asignación fase-máquina. Cada observación proporcionada al modelo debe corresponder a una decisión de asignación única. Mantener las imputaciones sin agregar provocaría que una misma fase apareciera representada en varias filas, otorgándole un peso artificialmente elevado en el entrenamiento y distorsionando el aprendizaje de los patrones de idoneidad.

La agregación se ha diseñado, además, preservando la información relevante contenida en las imputaciones individuales. Más allá de la suma de los tiempos, se conservan atributos constantes dentro de la fase (como la orden de fabricación o el material) y se derivan variables agregadas de interés analítico, garantizando que la reducción a un único registro por fase no suponga una pérdida de información útil para el modelado.

In [93]:
display(df_imput_productivas.head(0))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid


In [94]:
# # 1. me quedo solo con las imputaciones de las fases válidas
# imput_validas = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_validas)]

# # 2. máquina dominante de cada fase (el nodo con más tiempo real)
#     # Agrupa por la pareja fase + nodo, y suma el tiempo real de cada combinación.
# tiempo_fase_nodo = imput_validas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum().reset_index()
#     # Encontrar, por fase, la fila del tiempo máximo:
# idx_dom = tiempo_fase_nodo.groupby("SmFaseUid")["MinutosRealNbr"].idxmax()
#     # Recuperar esas filas y quedarse con fase + nodo
# dominante = tiempo_fase_nodo.loc[idx_dom, ["SmFaseUid", "smNodoId"]].rename(columns={"smNodoId": "MaquinaDominante"})

# # 3. tiempos agregados por fase (suma de TODAS sus imputaciones)
# tiempos = imput_validas.groupby("SmFaseUid").agg(
#     smOrdenId=("smOrdenId", "first"),
#     smMaterialId=("smMaterialId", "first"),
#     smFaseNme=("smFaseNme", "first"),
#     smOrdenNme=("smOrdenNme", "first"),
#     MinutosReal_total=("MinutosRealNbr", "sum"),
#     MinutosPreparacion_total=("MinutosPreparacionNbr", "sum"),
#     MinutosProduccion_total=("MinutosProduccionNbr", "sum"),
#     n_imputaciones=("MinutosRealNbr", "size"),
# ).reset_index()

# # 4. uno máquina dominante + tiempos en la tabla final (1 fila por fase)
# fase_agregada = dominante.merge(tiempos, on="SmFaseUid", how="left")

# print(f"Fases en la tabla final: {len(fase_agregada)}")
# display(fase_agregada.sample(5))

Tiempo bruto y tiempo efectivo: corrección por dedicación

El MES registra el tiempo de cada imputación como tiempo de reloj (el transcurrido entre inicio y fin), sin considerar la atención real dedicada a la tarea. Como es habitual que un operario gestione varias máquinas a la vez —abre una operación, atiende otra y regresa—, el reloj sigue corriendo aunque no le dedique el 100% de su atención. Para reflejarlo, el MES registra el porcentaje de dedicación de cada imputación, que se reparte entre las tareas concurrentes.

Se definen así dos magnitudes:

- Tiempo bruto (`MinutosReal_total`): el tiempo de reloj sin corregir.      
- Tiempo efectivo (`MinutosReal_efectivo`): el tiempo ponderado por la dedicación (tiempo × dedicación / 100), que representa el trabajo realmente invertido en la fase. 

>La corrección se aplica a nivel de imputación, antes de agregar.

La distinción es clave para la variable objetivo: el tiempo teórico del ERP asume dedicación plena, por lo que solo es comparable con el tiempo efectivo. Usar el bruto penalizaría injustamente a las asignaciones ejecutadas en multitarea, atribuyendo a la máquina un exceso de tiempo debido a una circunstancia organizativa ajena a su idoneidad. Por ello, el ratio se calcula sobre el tiempo efectivo, conservando el bruto solo para trazabilidad.

In [95]:
# 1. me quedo solo con las imputaciones de las fases válidas
imput_validas = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_validas)].copy()

# 1b. corrijo los tiempos por dedicación (tiempo de reloj → tiempo efectivo)
imput_validas["MinutosReal_efectivo"] = imput_validas["MinutosRealNbr"] * imput_validas["PorcentajeDedicacionNbr"] / 100
imput_validas["MinutosPreparacion_efectivo"] = imput_validas["MinutosPreparacionNbr"] * imput_validas["PorcentajeDedicacionNbr"] / 100
imput_validas["MinutosProduccion_efectivo"] = imput_validas["MinutosProduccionNbr"] * imput_validas["PorcentajeDedicacionNbr"] / 100

# 2. máquina dominante de cada fase (el nodo con más tiempo EFECTIVO)
tiempo_fase_nodo = imput_validas.groupby(["SmFaseUid", "smNodoId"])["MinutosReal_efectivo"].sum().reset_index()
idx_dom = tiempo_fase_nodo.groupby("SmFaseUid")["MinutosReal_efectivo"].idxmax()
dominante = tiempo_fase_nodo.loc[idx_dom, ["SmFaseUid", "smNodoId"]].rename(columns={"smNodoId": "MaquinaDominante"})

# 3. tiempos agregados por fase
tiempos = imput_validas.groupby("SmFaseUid").agg(
    smOrdenId=("smOrdenId", "first"),
    smMaterialId=("smMaterialId", "first"),
    smFaseNme=("smFaseNme", "first"),
    smOrdenNme=("smOrdenNme", "first"),
    # tiempos EFECTIVOS (corregidos por dedicación) → los del ratio
    MinutosReal_efectivo=("MinutosReal_efectivo", "sum"),
    MinutosPreparacion_efectivo=("MinutosPreparacion_efectivo", "sum"),
    MinutosProduccion_efectivo=("MinutosProduccion_efectivo", "sum"),
    # tiempos BRUTOS (los conservo para comparar / trazabilidad)
    MinutosReal_total=("MinutosRealNbr", "sum"),
    MinutosPreparacion_total=("MinutosPreparacionNbr", "sum"),
    MinutosProduccion_total=("MinutosProduccionNbr", "sum"),
    # dedicación como feature
    dedicacion_media=("PorcentajeDedicacionNbr", "mean"),
    n_imputaciones=("MinutosRealNbr", "size"),
).reset_index()

# 4. uno máquina dominante + tiempos en la tabla final (1 fila por fase)
fase_agregada = dominante.merge(tiempos, on="SmFaseUid", how="left")

# 5. verificaciones
print(f"Fases en la tabla final: {len(fase_agregada)}")
efectivo_supera = ((fase_agregada["MinutosReal_efectivo"] - fase_agregada["MinutosReal_total"]) > 0.01).sum()
print(f"Fases donde efectivo supera bruto de verdad (>0.01): {efectivo_supera}")  # debe ser 0
display(fase_agregada.sample(5))

Fases en la tabla final: 18548
Fases donde efectivo supera bruto de verdad (>0.01): 0


,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones
12640,ADE37729-A550-4042-B0F2-A6A6FA07D2D5,TN09,OF2605524,PE7JA3X0008,TPP,COPPER PART Holder MD335744L,65.20000,65.20000,0.00000,65.20000,65.20000,0.00000,66.666667,3
8486,7529845C-D54F-4289-BB1C-2F77D902E4EE,TN09,OF2512730,PE1XA3X0485,TPP,Portaelectrodos KAA36712S(IM00071369),87.35000,87.35000,0.00000,87.35000,87.35000,0.00000,50.000000,2
17987,F81ADA5C-219B-4A9E-975B-2DA677FA0F04,TP02,OF2600298,BR0XA3X0039,MTP,COPPER PART Arm MD154055S,38.56667,38.56667,0.00000,38.56667,38.56667,0.00000,50.000000,2
10935,97F2B1CD-4EAB-4538-9FDB-19292F0FFBD7,TN09,OF2600640,PE1XNCX0210,TPP,Elektrodenaufnahme F704501032201,23.10000,23.10000,0.00000,23.10000,23.10000,0.00000,50.000000,2
2228,1F376777-8030-49F9-91B9-F0C50D3E2D95,TN10,OF2506525,PA0XA2X0507,TCN,ADAPTATEUR JL4UWB00407100,84.26667,84.05000,0.21667,84.26667,84.05000,0.21667,50.000000,2


### Revisar el correcto traspaso de tiempos en las agrupaciones:
Los minutos totales se calcula de la siguiente manera:
`MinutosReal_total = MinutosPreparacion_total + MinutosProduccion_total`    
Entonces si calculo la diferencia entre Preparación y Producción debe dar 0


In [96]:
# ============================================================
# DEPURACIÓN: fases con tiempo inflado por MES sin cerrar
# ============================================================
# Criterio: un tiempo de preparación o producción desproporcionado
# (>= 480 min ≈ 1 turno) junto con un nº elevado de imputaciones
# indica que la máquina quedó abierta en el MES acumulando tiempo
# sin trabajo real.

UMBRAL_TIEMPO = 480          # minutos (1 turno de 8h) en prep o prod
UMBRAL_IMPUTACIONES = 3     # nº de imputaciones sospechoso

fase_agregada["mes_sin_cerrar"] = (
    (
        (fase_agregada["MinutosPreparacion_total"] >= UMBRAL_TIEMPO)
        | (fase_agregada["MinutosProduccion_total"] >= UMBRAL_TIEMPO)
    )
    & (fase_agregada["n_imputaciones"] > UMBRAL_IMPUTACIONES)
)

# reporte antes de eliminar
sospechosas = fase_agregada[fase_agregada["mes_sin_cerrar"]]
print(f"Fases marcadas como MES sin cerrar: {len(sospechosas)}")
print(sospechosas.nlargest(10, "MinutosProduccion_total")[
    ["SmFaseUid", "MaquinaDominante", "smFaseNme",
     "MinutosPreparacion_total", "MinutosProduccion_total", "n_imputaciones"]
].to_string(index=False))

antes = len(fase_agregada)
fase_agregada = fase_agregada[~fase_agregada["mes_sin_cerrar"]]
print(f"\nFases antes: {antes} | eliminadas: {antes - len(fase_agregada)} | después: {len(fase_agregada)}")

fase_agregada = fase_agregada.drop(columns=["mes_sin_cerrar"])

Fases marcadas como MES sin cerrar: 1159
                           SmFaseUid MaquinaDominante smFaseNme  MinutosPreparacion_total  MinutosProduccion_total  n_imputaciones
5D715E06-67F3-451C-AE5E-982E0083402F             CM04    CV-ALU                4968.85000              32412.53333              77
EDD89008-F84F-427B-802D-24BFEA25DEE4             CM17     MEC03                 585.76666              30921.84999              74
B1378B1E-8601-4B4F-A08C-501D8A0703E6             CM17     MEC03                 585.29999              30917.48332              74
BDD4BE9B-9BCF-4BCF-BCF4-E1BBB9E780DC             CM17     MEC03                 585.71666              30917.48332              74
1A5CBCD3-3A66-4C09-82A2-ADF8A9BE0E3A             CM17     MEC03                2243.95000              28929.40000              74
3A56FC28-C0C7-4219-9701-C91B548BB507             CM17     MEC03                2243.95000              28929.40000              74
68416ED9-521E-405A-8DB1-0D42D2CDDF84      

In [97]:
display(fase_agregada.sample(3))

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones
455,05FB889D-02C2-46A5-885E-CD96DB254E7A,CM08,OF2602461,BALX60X0110,CV-BAL,Braç d'Alumini 5645053,17.73333,17.73333,0.0,17.73333,17.73333,0.0,50.0,2
9863,895A1AA2-42B1-4538-9755-1B9E4EDD1AB0,TP02,OF2511028,CODXA2X0020,MTP,Strombrücke A8043035,152.85000,152.85000,0.0,152.85000,152.85000,0.0,50.0,2
2397,219D670E-9218-4DE8-AC5D-3D808DD2FDC6,CM06,OF2507800,PE1XA2X0690,PE01,Portaelectrodes Ø30 7091501755001,31.75000,31.75000,0.0,31.75000,31.75000,0.0,50.0,2


> Hay imputaciones con un ruido inmanejable, voy a tener que eliminarlas.

Durante la validación de los tiempos registrados en el sistema de captura de planta (MES) se observó, en un subconjunto de fases, una incoherencia entre el tiempo real imputado y la suma de sus componentes de preparación y producción. Al analizar estos casos, se identificó que las desviaciones más severas se concentraban en fases con un número anormalmente elevado de imputaciones. La inspección manual de estas fases reveló el origen del problema: en determinadas ocasiones, una máquina permanecía con la orden abierta en el MES tras finalizar el trabajo efectivo, de modo que el sistema continuaba acumulando imputaciones de tiempo real de forma automática, sin actividad productiva real asociada. Este tiempo espurio inflaba de forma muy acusada el tiempo real de la fase.

Dado que el tiempo real constituye el numerador de la variable objetivo del estudio (el ratio entre tiempo real y teórico), estos registros habrían distorsionado gravemente el cálculo de la idoneidad, atribuyendo desviaciones elevadas a asignaciones que en realidad fueron correctas y cuyo exceso de tiempo se debía únicamente a un fallo en el cierre del registro. Por este motivo, y tras verificar individualmente su naturaleza, se han descartado las fases afectadas por este fenómeno. El criterio de identificación combinó la magnitud de la desviación con el número de imputaciones de la fase, lo que permitió aislar los registros defectuosos sin eliminar aquellas fases cuya ligera desviación responde a las condiciones normales de imputación del trabajo. Tras esta depuración, el conjunto de análisis queda constituido por 17.326 fases con tiempos fiables.

In [98]:
# Voy añadir el grupo de la máquina a la tabla final de imputaciones agregadas por fase
df_fases_MES = fase_agregada.merge(maquina_grupo, left_on="MaquinaDominante", right_on="CodigoMaquina", how="left")

In [99]:
# Verificar si alguna fase se quedo sin grupo máquina:
print(f"Fases sin grupo tras el merge: {df_fases_MES['NombreGrupo'].isna().sum()}")

Fases sin grupo tras el merge: 0


In [100]:
df_fases_MES = df_fases_MES.drop("CodigoMaquina", axis=1)

display(df_fases_MES.sample(1))
display(f"Grupos de máquinas: {len(df_fases_MES["CodigoGrupo"].unique())}")

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
15840,E8C3EAD0-7A04-4482-ACAC-D44F4AD1F71F,TN07,OF2507669,PA0XNCX0146,TCN,Elektrodenschaft F704510530448,203.38334,203.16667,0.21667,203.38334,203.16667,0.21667,50.0,2,38a31fd3-7516-44fd-b395-24d834e43d2b,HYUNDAI-KIA SKT21 LMS Torn C.N.C,id_tcn,TCN,Torns Control Numèric


'Grupos de máquinas: 9'

In [101]:
df_fases_MES.to_parquet("res//procesed/df_fases_MES.parquet")